# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access the metadata as an object and print name and description (as attributes, not subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, record sets contain the data rows, and fields define the columns. We'll display available record sets and their associated field `@id`s for exploration.

In [ ]:
# List all record sets, fields, and columns by `@id`
print("Available record sets in this dataset:")
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set.id}")
    field_ids = [field.id for field in record_set.fields]
    print(f"  Fields (@id): {field_ids}\n")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. We'll extract the primary survey table and give an overview of its available fields, referencing by their `@id`.

In [ ]:
# Select all available record set @ids for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nData loaded for record set {record_set_id}.")
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head(2))

# If only one record set, select it for analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nSelected main record set for further analysis: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section illustrates: removing outliers, transforming distributions, and grouping data. All fields are referenced by their `@id`.

In [ ]:
#--- EDA Section ---#
# We'll use the PHQ-9 total score as a numeric column if present.
# Replace `phq9_total_score` and `village` below with exact field @id values after step 2 if your dataset uses other names.

# Reference the main DataFrame
df = dataframes[main_record_set_id]

# Identify a numeric field for filtering, e.g., PHQ-9 score, GAD-7 score, or PSQ
num_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['phq', 'gad', 'psq', 'score', 'age']) and pd.api.types.is_numeric_dtype(df[col])]

if num_field_candidates:
    numeric_field_id = num_field_candidates[0]
    print(f"Using numeric field: {numeric_field_id} for analysis.")
else:
    # Fallback to first numeric column if available
    numeric_field_id = df.select_dtypes('number').columns[0] if not df.empty else None

if numeric_field_id and not df[numeric_field_id].isnull().all():
    # Example threshold for demonstration
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (Q3 quantile):")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

    # Try grouping by a categorical field, e.g., 'village' or 'marital_status'
    group_field_candidates = [col for col in df.columns if col.lower() in ['village', 'marital_status', 'level_of_education', 'gender']]
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id}, displaying mean {numeric_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use pandas built-in plotting (with matplotlib) to show basic histograms and boxplots. Adjust your field `@id` as identified above.

In [ ]:
import matplotlib.pyplot as plt

# Plot distribution of the main numeric field
if numeric_field_id:
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if group_field_id exists)
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded using the `mlcroissant` library and explored by referencing all entities by their `@id` as per best practices for Croissant datasets.
- We identified the available record sets and fields. Typical numeric fields such as PHQ-9 or GAD-7 summary scores were used for demonstration of filtering, normalization, and grouping.
- Visualizations provide a quick perspective on score distributions and group differences (e.g., by village, gender, or education).
- Further work may involve detailed statistical analysis or prediction modeling using the extracted fields.

**Note:** Always consult the dataset's metadata and documentation for field explanations and ethical considerations before deeper use.